# Customer model validation

Run from the repository root with the venv activated and `pip install -e .` so `app` resolves.

In [ ]:
from app.domain.customer import Customer, PurchaseContext
import numpy as np

rng = np.random.default_rng(42)
c_new = Customer(
    customer_id=1,
    customer_index=0,
    budget=50.0,
    buy_propensity=0.3,
    price_threshold=25.0,
    repeat_boost=0.4,
    basket_mean=18.0,
    location_zone="A",
)
c_ret = Customer(
    customer_id=2,
    customer_index=1,
    budget=50.0,
    buy_propensity=0.3,
    price_threshold=25.0,
    repeat_boost=0.4,
    basket_mean=18.0,
    location_zone="B",
    has_purchased_before=True,
    purchase_count=2,
)
ctx = PurchaseContext(temporal_multiplier=1.0, geographic_multiplier=1.0, promo_eligible=True)

p_high = c_new.compute_purchase_probability(60.0, 1, ctx)
p_low = c_new.compute_purchase_probability(20.0, 1, ctx)
assert p_high == 0.0, "over budget should zero out"
assert 0 <= p_low <= 1

p_ret = c_ret.compute_purchase_probability(20.0, 5, ctx)
assert p_ret >= p_low, "returning customer should be easier to convert"
print("budget block", p_high, "new @ low price", p_low, "returning @ low", p_ret)

In [ ]:
n = 2000
buys = sum(c_new.decide_purchase(rng, 22.0, 1, ctx) for _ in range(n))
print("empirical purchase rate new", buys / n)